In [ ]:
# Retrieval-Augmented Generation (RAG) System

# Phase 1: Environment Setup & Dependencies
# In this section, we configure the environment, suppress unnecessary warnings for a clean output log, and initialize the required LangChain, FAISS,
# and Google Gemini modules. We also securely load the API key.

In [1]:
# --- 1. System & Logging Configuration ---
import os
import logging
import warnings
import time

# Suppress TensorFlow and HuggingFace warnings for cleaner execution logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# Хак для TensorFlow (щоб прибрати помилку losses)
try:
    import tensorflow as tf
    tf.get_logger().setLevel(logging.ERROR)
    tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
except ImportError:
    pass

# --- 2. Core RAG Libraries ----
import getpass
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# --- 3. API Key Authentication ---
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Введи Google API Key: ")

print("✅ Всі бібліотеки та налаштування завантажено коректно!")

Введи Google API Key:  ········


✅ Всі бібліотеки та налаштування завантажено коректно!


In [ ]:
# Phase 2: Data Ingestion & Vector Space Modeling
# We load the corpus (e.g., an article on Japanese Art), clean the raw text using Regular Expressions, and chunk it into semantic segments. 
# These chunks are then converted into dense vector embeddings using a multilingual HuggingFace model and 
# indexed in a FAISS vector database for fast similarity search.

In [ ]:
import re 

file_path = "data.txt"

try:
    # --- 1. Data Loading ---
    print(f"📖 Читаю файл '{file_path}'...")
    loader = TextLoader(file_path, encoding='utf-8')
    raw_documents = loader.load()
    
    # --- 2. Text Cleaning (Noise Reduction) ---
    content = raw_documents[0].page_content
    
    # Replace multiple spaces/newlines with a single space
    content = re.sub(r'\s+', ' ', content).strip()
    raw_documents[0].page_content = content
    
    print(f"🧹 Текст очищено від шуму! Символів: {len(content)}")
    # ========================================================

    # --- 3. Semantic Chunking ---
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = text_splitter.split_documents(raw_documents)
    
    # Remove potential duplicate chunks
    unique_chunks = []
    seen_content = set()
    for chunk in chunks:
        if chunk.page_content not in seen_content:
            unique_chunks.append(chunk)
            seen_content.add(chunk.page_content)
    
    chunks = unique_chunks
    
    print(f"✂️ Текст розбито на {len(chunks)} унікальних фрагментів.")

    # --- 4. Vector Embedding & Indexing ---
    print("🧠 Генерую вектори (FAISS)...")
    embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
    vector_db = FAISS.from_documents(chunks, embeddings)
    print("✅ Векторна база готова!")

except FileNotFoundError:
    print(f"❌ ПОМИЛКА: Файл '{file_path}' не знайдено!")
except Exception as e:
    print(f"❌ Сталася помилка: {e}")

In [ ]:
# Phase 3: Conversational RAG Pipeline
# We initialize the Google Gemini LLM and construct a robust chain that integrates chat history, semantic retrieval, and prompt formatting. 
# We also include error handling to gracefully manage API rate limits.

In [15]:
# Initialize LLM (Using standard Gemini 1.5 Flash for speed and reliability)
llm = ChatGoogleGenerativeAI(model="gemini-exp-1206", temperature=0)

chat_history = []

def chat_with_rag(user_query):
    # --- 1. Retrieval Phase ---
    results = vector_db.similarity_search_with_score(user_query, k=5)
    
    print(f"\n🔍 [RAG] Знайдено контекст для запиту: '{user_query}'")
    
    # Optional: Print retrieval details for debugging
    # print(f"\n🔍 [Retrieval] Top matches for: '{user_query}'")
    # for i, (doc, score) in enumerate(results, 1):
    #     print(f"   {i}. [Score: {score:.4f}] {doc.page_content[:80]}...") 

    # --- 2. Prompt Construction ---
    system_prompt = """Ти корисний асистент. Відповідай українською мовою.
    Використовуй ТІЛЬКИ наданий контекст.
    Контекст: {context}
    """
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}")
    ])

    chain = prompt | llm
    
    # Prevent aggressive API rate limiting
    time.sleep(2)

    # --- 3. Generation Phase (with Fallbacks) ---
    answer_text = ""
    try:
        response = chain.invoke({
            "context": context_text,
            "history": chat_history,
            "question": user_query
        })
        answer_text = response.content
        
    except Exception as e:
        err_msg = str(e)
        if "429" in err_msg or "RESOURCE_EXHAUSTED" in err_msg:
            answer_text = "⚠️ (Google Limit) Ліміт запитів вичерпано. Але подивіться вище - ПОШУК (Retrieval) спрацював успішно!"
        elif "404" in err_msg:
            answer_text = "⚠️ (Model Error) Модель недоступна. Але ПОШУК (Retrieval) спрацював успішно!"
        else:
            answer_text = f"⚠️ Технічна помилка API: {e}"

    # Update conversation history
    chat_history.append(HumanMessage(content=user_query))
    chat_history.append(AIMessage(content=answer_text))
    
    return answer_text

print("✅ RAG-система готова! Ввімкнено захист від збоїв API.")

✅ RAG-система готова! Ввімкнено захист від збоїв API.


In [ ]:
# Phase 4: System Evaluation
# To validate the system's accuracy, retrieval quality, and hallucination resistance, we execute a comprehensive test suite. 
# This includes factual queries based on the corpus, as well as "trick" questions designed to test the system's strict adherence to the provided context.

In [4]:
test_questions = [
    # --- Block 1: History & Art ---
    "Які три релігійно-філософські доктрини вплинули на японське мистецтво?",
    "Що таке період Дзьомон і чому він так називається?",
    "Хто вигадав слово 'манга' і що воно означає?",
    "Що таке нецке і для чого вони використовувались?",
    
    # --- Block 2: Architecture & Gardens ---
    "Чому деревина стала головним будівельним матеріалом у японській архітектурі?",
    "Що символізують білий пісок і гравій у релігії Синто та садах дзен?",
    "Скільки каменів у саду Рьоан-дзі і на скільки груп вони поділені?",
    "Як називається перше керівництво по створенню японських садів?",
    "Які правила розташування каменів описує книга Sakuteiki?",
    
    # --- Block 3: Theatrical Arts ---
    "Чим відрізняються актори театру Кьоген від акторів театру Но?",
    "Чому в театрі Кабукі жіночі ролі почали виконувати чоловіки?",
    "Якого розміру ляльки використовуються в театрі Бунраку?",
    "Хто така Оксана Степанюк?",

    # --- Block 4: Hallucination Checks (Context Boundary Testing) ---
    "Яка точна висота гори Фудзі в метрах?", # System should state it doesn't know based on context
    "Як правильно приготувати суші?"         # Completely out of context
]

print(f"--- ЗАПУСК ВЕЛИКОГО ТЕСТУ ({len(test_questions)} питань) ---\n")

for i, question in enumerate(test_questions, 1):
    print(f"🔹 Питання {i}: {question}")
    
    # Виклик RAG
    answer = chat_with_rag(question)
    
    print(f"🤖 Відповідь: {answer}")
    print("-" * 50)

--- ЗАПУСК ВЕЛИКОГО ТЕСТУ (15 питань) ---

🔹 Питання 1: Які три релігійно-філософські доктрини вплинули на японське мистецтво?

🔍 [RAG] Знайдено контекст для запиту: 'Які три релігійно-філософські доктрини вплинули на японське мистецтво?'
   1. [Score: 6.3060] Традиційне японське мистецтво екологічне і медитативне, у ньому значна увага надається відстороненом...
   2. [Score: 10.7798] . Три Королівства, і особливо Бекдже, мали важливий активний вплив на впровадження і формування будд...
   3. [Score: 11.1414] . Орієнтування японців на особистісні фактори — визначальна риса мистецтва, яка обумовлена буддизмом...
🤖 Відповідь: Естетичні принципи японського мистецтва сформувалися під впливом трьох найважливіших релігійно-філософських доктрин: синтоїзму, конфуціанства і буддизму.
--------------------------------------------------
🔹 Питання 2: Що таке період Дзьомон і чому він так називається?

🔍 [RAG] Знайдено контекст для запиту: 'Що таке період Дзьомон і чому він так називається?'
   1. [

In [11]:
# --- Additional: file diagnosis ---
keyword = "театр" # або "Кабукі"

# Check if this word is in the loaded text
found = False
count = 0

print(f"🔎 Шукаю слово '{keyword}' у завантажених чанках...")

for i, chunk in enumerate(chunks):
    if keyword.lower() in chunk.page_content.lower():
        print(f"✅ Знайдено у чанку №{i}: {chunk.page_content[:100]}...")
        found = True
        count += 1

if not found:
    print(f"❌ СЛОВО '{keyword}' НЕ ЗНАЙДЕНО В БАЗІ!")
    print("Причина: Швидше за все, ти вставив текст у файл data.txt, але НЕ ЗБЕРІГ його (Ctrl+S).")
    print("Або файл завантажився не повністю.")
else:
    print(f"Знайдено згадувань: {count}. Дивно, чому пошук його не видав...")

🔎 Шукаю слово 'театр' у завантажених чанках...
✅ Знайдено у чанку №0: Мистецтво Японії
Мисте́цтво Япо́нії — сукупність культурного надбання японського народу за період — ...
✅ Знайдено у чанку №16: . Час виявляв сутність предметів. Наприклад, керамічна чаша для чайної церемонії зберігала сліди дот...
✅ Знайдено у чанку №20: . У них проявлявся інтерес до драматичних ситуацій і конкретних подій. Герої емакімоно дотримувались...
✅ Знайдено у чанку №26: . Відділ з питань культури агентства поширює інформацію про мистецтво в Японії та на міжнародному рі...
✅ Знайдено у чанку №41: У театральному житті Японії традиційні театральні жанри — ноо (але, ногаку), кабуки, ляльковий театр...
✅ Знайдено у чанку №42: . Нерухомістю немовби вимикали себе з поля зору глядачів. У XVI—XVII столітті головним жанром японсь...
✅ Знайдено у чанку №46: . Однак сценічне мистецтво, як правило, поважали менш, і нібито аморальність актрис раннього театру ...
✅ Знайдено у чанку №49: Багато традиційних форм японської 